# SQS суперячейки 64 атома: Al-Co-Cr-Fe-Ni-Ti (L12)
Упрочняющая фаза **(Ni,Co,Fe,Cr)₃(Al,Ti)** со структурой **L12** (Pm-3m, #221)

Генерация **Special Quasirandom Structure (SQS)** с помощью **sqsgenerator**.
- Размер: **64 атома** (суперячейка 2×2×4)
- Режим: `sublattice_mode: split` (строгое разделение A/B подрешёток)
- Итерации: **50 000 000**

In [1]:
%pip install sqsgenerator pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.8 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609

In [2]:
from sqsgenerator import parse_config, optimize, write
from pymatgen.core import Structure
import re

## 1. Параметры и генерация координат

In [3]:
a = 3.57
nx, ny, nz = 2, 2, 4  # 2x2x4 -> 16 ячеек * 4 атома = 64 атома

coords = []
species = []

for ix in range(nx):
    for iy in range(ny):
        for iz in range(nz):
            # B-сайт (1a)
            coords.append([ix/nx, iy/ny, iz/nz])
            species.append("Al")
            # A-сайты (3c)
            for offset in [[0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]]:
                coords.append([(ix+offset[0])/nx, (iy+offset[1])/ny, (iz+offset[2])/nz])
                species.append("Ni")

print(f"Сгенерировано сайтов: {len(coords)}")

Сгенерировано сайтов: 64


## 2. Конфигурация sqsgenerator

In [4]:
config = {
    "iterations": 50000000,
    "sublattice_mode": "split",
    "structure": {
        "lattice": [[a*nx, 0, 0], [0, a*ny, 0], [0, 0, a*nz]],
        "coords": coords,
        "species": species,
        "supercell": [1, 1, 1]
    },
    "composition": [
        # B-подрешётка (16 сайтов): Al:Ti = 3:1 -> 12 Al, 4 Ti
        {"sites": "Al", "Al": 12, "Ti": 4},
        # A-подрешётка (48 сайтов): Ni:Co:Fe:Cr = 12:5:4:3 -> 24 Ni, 10 Co, 8 Fe, 6 Cr
        {"sites": "Ni", "Ni": 24, "Co": 10, "Fe": 8, "Cr": 6}
    ],
    "max_results_per_objective": 5
}

print("Конфигурация готова. Запуск оптимизации...")

Конфигурация готова. Запуск оптимизации...


## 3. Запуск оптимизации SQS

In [5]:
cfg = parse_config(config)
pack = optimize(cfg)

print(f"\nНайдено решений: {len(pack)}")
print("Топ-5 конфигураций по Objective (ниже = лучше):")
for i in range(min(5, len(pack))):
    obj, sols = pack[i]
    print(f"  #{i}: {obj:.6f} ({len(sols)} структур)")


Найдено решений: 40
Топ-5 конфигураций по Objective (ниже = лучше):
  #0: 10.297500 (1 структур)
  #1: 10.327083 (1 структур)
  #2: 10.435833 (1 структур)
  #3: 10.553750 (1 структур)
  #4: 10.625000 (1 структур)


## 4. Экспорт и очистка CIF

In [9]:
best = pack.best()
print(f"Лучший objective: {best.objective:.6f}")

# Экспорт в CIF
write(best.structure(), "HEA_L12_SQS_64atoms.cif")

Лучший objective: 10.297500


In [11]:
!sed -i 's/0+//g' HEA_L12_SQS_64atoms.cif

## 5. Проверка структуры

In [12]:
sqs_64 = Structure.from_file("HEA_L12_SQS_64atoms.cif")

print(f"Формула: {sqs_64.composition.formula}")
print(f"Число атомов: {sqs_64.num_sites}")
print(f"Решётка: {sqs_64.lattice.a:.2f} x {sqs_64.lattice.b:.2f} x {sqs_64.lattice.c:.2f}")
print(f"\nСостав:")
for el, amt in sqs_64.composition.as_dict().items():
    print(f"  {el}: {amt:.0f} ({amt/sqs_64.num_sites*100:.1f}%)")

Формула: Ti4 Al12 Cr6 Fe8 Co10 Ni24
Число атомов: 64
Решётка: 7.14 x 7.14 x 14.28

Состав:
  Ti0+: 4 (6.2%)
  Al0+: 12 (18.8%)
  Cr0+: 6 (9.4%)
  Fe0+: 8 (12.5%)
  Co0+: 10 (15.6%)
  Ni0+: 24 (37.5%)
